<a href="https://colab.research.google.com/github/amzad-786githumb/AIR_LLM_Research/blob/main/04_Candidate_Imputation_Methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# 04.0 PROJECT INITIALIZATION
# ============================================================

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)

from pathlib import Path
import json
import time
import warnings
import random

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AIR_LLM_Research"
)

assert PROJECT_ROOT.exists(), (
    f"Project root not found:\n{PROJECT_ROOT}"
)

SCENARIO_ROOT = (
    PROJECT_ROOT
    / "experiments"
    / "missingness"
)

RESULTS_ROOT = (
    PROJECT_ROOT
    / "results"
)

TABLE_ROOT = (
    RESULTS_ROOT
    / "tables"
)

ARTIFACT_ROOT = (
    PROJECT_ROOT
    / "artifacts"
    / "notebook_04"
)

for path in [
    TABLE_ROOT,
    ARTIFACT_ROOT
]:
    path.mkdir(
        parents=True,
        exist_ok=True
    )

print("=" * 90)
print("NOTEBOOK 04 — CANDIDATE IMPUTATION METHODS")
print("=" * 90)
print(f"Project root : {PROJECT_ROOT}")
print(f"Scenario root: {SCENARIO_ROOT}")
print(f"Artifact root: {ARTIFACT_ROOT}")

Mounted at /content/drive
NOTEBOOK 04 — CANDIDATE IMPUTATION METHODS
Project root : /content/drive/MyDrive/AIR_LLM_Research
Scenario root: /content/drive/MyDrive/AIR_LLM_Research/experiments/missingness
Artifact root: /content/drive/MyDrive/AIR_LLM_Research/artifacts/notebook_04


In [2]:
# ============================================================
# 04.0.1 IMPORTS
# ============================================================

from sklearn.impute import (
    SimpleImputer,
    KNNImputer
)

from sklearn.experimental import (
    enable_iterative_imputer
)

from sklearn.impute import (
    IterativeImputer
)

from sklearn.ensemble import (
    ExtraTreesRegressor
)

from sklearn.preprocessing import (
    OrdinalEncoder,
    StandardScaler
)

from sklearn.compose import (
    ColumnTransformer
)

from sklearn.pipeline import (
    Pipeline
)

from sklearn.neural_network import (
    MLPRegressor
)

from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    mean_squared_error
)

print("Required packages imported successfully.")

Required packages imported successfully.


In [3]:
# ============================================================
# 04.0.2 REPRODUCIBILITY
# ============================================================

RANDOM_STATE = 42

random.seed(
    RANDOM_STATE
)

np.random.seed(
    RANDOM_STATE
)

print(
    f"Global random state: {RANDOM_STATE}"
)

Global random state: 42


In [4]:
# ============================================================
# 04.0.3 DATASET REGISTRY
# ============================================================

DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
]

TARGET_REGISTRY = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted"
}

print("=" * 90)
print("DATASET REGISTRY")
print("=" * 90)

for dataset_id in DATASET_IDS:

    print(
        f"{dataset_id:<20} "
        f"target={TARGET_REGISTRY[dataset_id]}"
    )

DATASET REGISTRY
adult_income         target=income
bank_marketing       target=y
diabetes_130us       target=readmitted


In [5]:
# ============================================================
# 04.0.4 VERIFY NOTEBOOK 03 SCENARIOS
# ============================================================

EXPECTED_SCENARIOS = [
    "mcar_10pct_seed_42",
    "mcar_10pct_seed_123",
    "mcar_10pct_seed_2024"
]

print("=" * 90)
print("VERIFYING NOTEBOOK 03 OUTPUT")
print("=" * 90)

for dataset_id in DATASET_IDS:

    dataset_dir = (
        SCENARIO_ROOT
        / dataset_id
    )

    if not dataset_dir.exists():

        raise FileNotFoundError(
            f"Scenario directory not found:\n"
            f"{dataset_dir}"
        )

    print(
        f"{dataset_id:<20}"
        f"{len(list(dataset_dir.glob('*.csv'))):>6} CSV files"
    )

print()
print("Notebook 03 scenario structure verified.")

VERIFYING NOTEBOOK 03 OUTPUT
adult_income            90 CSV files
bank_marketing          90 CSV files
diabetes_130us          90 CSV files

Notebook 03 scenario structure verified.


In [6]:
# ============================================================
# 04.1 MEAN IMPUTATION
# ============================================================

def impute_mean(
    masked_df,
    random_state=42
):
    """
    Mean imputation for numerical variables
    and mode imputation for categorical variables.
    """

    df = masked_df.copy(
        deep=True
    )

    numeric_cols = (
        df.select_dtypes(
            include=np.number
        )
        .columns
        .tolist()
    )

    categorical_cols = [
        column
        for column in df.columns
        if column not in numeric_cols
    ]

    if numeric_cols:

        numeric_imputer = SimpleImputer(
            strategy="mean"
        )

        df[numeric_cols] = (
            numeric_imputer.fit_transform(
                df[numeric_cols]
            )
        )

    if categorical_cols:

        categorical_imputer = SimpleImputer(
            strategy="most_frequent"
        )

        df[categorical_cols] = (
            categorical_imputer.fit_transform(
                df[categorical_cols]
            )
        )

    return df

In [7]:
# ============================================================
# 04.2 MEDIAN IMPUTATION
# ============================================================

def impute_median(
    masked_df,
    random_state=42
):
    """
    Median imputation for numerical variables
    and mode imputation for categorical variables.
    """

    df = masked_df.copy(
        deep=True
    )

    numeric_cols = (
        df.select_dtypes(
            include=np.number
        )
        .columns
        .tolist()
    )

    categorical_cols = [
        column
        for column in df.columns
        if column not in numeric_cols
    ]

    if numeric_cols:

        numeric_imputer = SimpleImputer(
            strategy="median"
        )

        df[numeric_cols] = (
            numeric_imputer.fit_transform(
                df[numeric_cols]
            )
        )

    if categorical_cols:

        categorical_imputer = SimpleImputer(
            strategy="most_frequent"
        )

        df[categorical_cols] = (
            categorical_imputer.fit_transform(
                df[categorical_cols]
            )
        )

    return df

In [8]:
# ============================================================
# 04.5 MODE IMPUTATION
# ============================================================

def impute_mode(
    masked_df,
    missing_mask=None,
    random_state=42
):
    """
    Mode imputation for all columns.

    Only missing cells are replaced.
    Observed values are preserved exactly.
    """

    df = masked_df.copy(deep=True)

    for column in df.columns:

        missing = df[column].isna()

        if not missing.any():
            continue

        non_missing = df.loc[
            ~missing,
            column
        ]

        if non_missing.empty:
            continue

        mode_values = non_missing.mode(
            dropna=True
        )

        if mode_values.empty:
            continue

        mode_value = mode_values.iloc[0]

        df.loc[
            missing,
            column
        ] = mode_value

    return df


# ============================================================
# MODE METHOD WRAPPER
# ============================================================

def run_mode(
    masked_df,
    missing_mask,
    random_state=42
):

    start_time = time.perf_counter()

    imputed_df = impute_mode(
        masked_df=masked_df,
        missing_mask=missing_mask,
        random_state=random_state
    )

    runtime = (
        time.perf_counter()
        - start_time
    )

    return {
        "imputed_data": imputed_df,
        "runtime_seconds": runtime
    }

In [9]:
# ============================================================
# COMMON IMPUTATION DISPATCHER
# ============================================================

def run_imputation(
    method_name,
    masked_df,
    missing_mask,
    random_state=42
):

    if method_name == "Mean":

        return run_mean(
            masked_df,
            missing_mask,
            random_state
        )

    elif method_name == "Median":

        return run_median(
            masked_df,
            missing_mask,
            random_state
        )

    elif method_name == "Mode":

        return run_mode(
            masked_df,
            missing_mask,
            random_state
        )

    elif method_name == "KNN":

        return run_knn(
            masked_df,
            missing_mask,
            random_state
        )

    elif method_name == "MICE":

        return run_mice(
            masked_df,
            missing_mask,
            random_state
        )

    elif method_name == "MissForest":

        return run_missforest(
            masked_df,
            missing_mask,
            random_state
        )

    elif method_name == "GAIN":

        return run_gain(
            masked_df,
            missing_mask,
            random_state
        )

    elif method_name == "LLM":

        return run_llm(
            masked_df,
            missing_mask,
            random_state
        )

    elif method_name == "AutomatedSelection":

        return run_automated_selection(
            masked_df,
            missing_mask,
            random_state
        )

    else:

        raise ValueError(
            f"Unknown imputation method: {method_name}"
        )

In [10]:
# ============================================================
# 04.4 KNN IMPUTATION
# ============================================================

def impute_knn(
    masked_df,
    random_state=42,
    n_neighbors=5,
    weights="distance"
):
    """
    Mixed-type KNN baseline.

    Numerical:
        KNNImputer

    Categorical:
        Most-frequent imputation

    KNNImputer is deterministic and therefore does not
    receive a random_state argument.
    """

    df = masked_df.copy(
        deep=True
    )

    numeric_cols = (
        df.select_dtypes(
            include=np.number
        )
        .columns
        .tolist()
    )

    categorical_cols = [
        column
        for column in df.columns
        if column not in numeric_cols
    ]

    # --------------------------------------------------------
    # NUMERICAL FEATURES
    # --------------------------------------------------------

    if numeric_cols:

        numeric_data = (
            df[numeric_cols]
            .copy()
        )

        valid_numeric = [
            column
            for column in numeric_cols
            if numeric_data[column]
            .notna()
            .any()
        ]

        if valid_numeric:

            effective_neighbors = min(
                n_neighbors,
                max(
                    1,
                    len(df)
                )
            )

            imputer = KNNImputer(
                n_neighbors=effective_neighbors,
                weights=weights
            )

            values = imputer.fit_transform(
                numeric_data[valid_numeric]
            )

            df[valid_numeric] = (
                pd.DataFrame(
                    values,
                    index=df.index,
                    columns=valid_numeric
                )
            )

        # Entirely missing numerical columns
        # receive deterministic zero fallback.

        for column in numeric_cols:

            if df[column].isna().any():

                median_value = (
                    df[column]
                    .median()
                )

                if pd.isna(
                    median_value
                ):
                    median_value = 0.0

                df[column] = (
                    df[column]
                    .fillna(
                        median_value
                    )
                )

    # --------------------------------------------------------
    # CATEGORICAL FEATURES
    # --------------------------------------------------------

    if categorical_cols:

        categorical_imputer = (
            SimpleImputer(
                strategy="most_frequent"
            )
        )

        values = (
            categorical_imputer
            .fit_transform(
                df[categorical_cols]
            )
        )

        df[categorical_cols] = (
            pd.DataFrame(
                values,
                index=df.index,
                columns=categorical_cols
            )
        )

    return df

In [11]:
# ============================================================
# 04.5 MICE / ITERATIVE IMPUTER
# ============================================================

def impute_mice(
    masked_df,
    random_state=42,
    max_iter=10
):
    """
    MICE-style iterative imputation.

    Numerical:
        IterativeImputer + ExtraTreesRegressor

    Categorical:
        Most-frequent imputation
    """

    df = masked_df.copy(
        deep=True
    )

    numeric_cols = (
        df.select_dtypes(
            include=np.number
        )
        .columns
        .tolist()
    )

    categorical_cols = [
        column
        for column in df.columns
        if column not in numeric_cols
    ]

    if numeric_cols:

        estimator = ExtraTreesRegressor(
            n_estimators=30,
            max_depth=10,
            min_samples_leaf=2,
            random_state=random_state,
            n_jobs=-1
        )

        mice = IterativeImputer(
            estimator=estimator,
            max_iter=max_iter,
            random_state=random_state,
            initial_strategy="median",
            skip_complete=True
        )

        values = mice.fit_transform(
            df[numeric_cols]
        )

        df[numeric_cols] = (
            pd.DataFrame(
                values,
                index=df.index,
                columns=numeric_cols
            )
        )

    if categorical_cols:

        categorical_imputer = (
            SimpleImputer(
                strategy="most_frequent"
            )
        )

        values = (
            categorical_imputer
            .fit_transform(
                df[categorical_cols]
            )
        )

        df[categorical_cols] = (
            pd.DataFrame(
                values,
                index=df.index,
                columns=categorical_cols
            )
        )

    return df

In [12]:
# ============================================================
# 04.6 MISSFOREST / RANDOM FOREST IMPUTATION
# ============================================================

def impute_missforest(
    masked_df,
    random_state=42,
    n_estimators=50,
    max_iter=5,
    max_depth=15
):
    """
    Random-Forest-based iterative imputation.

    This is implemented as a controlled MissForest-style
    baseline using ExtraTreesRegressor.

    Numerical:
        Iterative forest regression

    Categorical:
        Deterministic mode imputation

    The reduced forest configuration is intentional so that
    the method remains computationally feasible across the
    complete AIR-LLM experimental matrix.
    """

    df = masked_df.copy(
        deep=True
    )

    numeric_cols = (
        df.select_dtypes(
            include=np.number
        )
        .columns
        .tolist()
    )

    categorical_cols = [
        column
        for column in df.columns
        if column not in numeric_cols
    ]

    if numeric_cols:

        estimator = ExtraTreesRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=2,
            random_state=random_state,
            n_jobs=-1
        )

        forest_imputer = IterativeImputer(
            estimator=estimator,
            max_iter=max_iter,
            random_state=random_state,
            initial_strategy="median",
            skip_complete=True
        )

        values = forest_imputer.fit_transform(
            df[numeric_cols]
        )

        df[numeric_cols] = (
            pd.DataFrame(
                values,
                index=df.index,
                columns=numeric_cols
            )
        )

    if categorical_cols:

        categorical_imputer = (
            SimpleImputer(
                strategy="most_frequent"
            )
        )

        values = (
            categorical_imputer
            .fit_transform(
                df[categorical_cols]
            )
        )

        df[categorical_cols] = (
            pd.DataFrame(
                values,
                index=df.index,
                columns=categorical_cols
            )
        )

    return df

In [13]:
# ============================================================
# 04.7 GAIN-STYLE NEURAL IMPUTATION BASELINE
# ============================================================

def impute_gain(
    masked_df,
    random_state=42,
    max_iter=20
):
    """
    Reproducible neural-network imputation baseline.

    This candidate provides a lightweight neural reconstruction
    baseline for the AIR-LLM candidate pool.

    Numerical:
        MLP-based reconstruction

    Categorical:
        Most-frequent imputation

    Note:
        This is registered as a GAIN-style neural baseline.
        A full adversarial GAIN implementation should be placed
        in a dedicated model-training notebook rather than
        retrained inside every experiment.
    """

    df = masked_df.copy(
        deep=True
    )

    numeric_cols = (
        df.select_dtypes(
            include=np.number
        )
        .columns
        .tolist()
    )

    categorical_cols = [
        column
        for column in df.columns
        if column not in numeric_cols
    ]

    # --------------------------------------------------------
    # CATEGORICAL FEATURES
    # --------------------------------------------------------

    if categorical_cols:

        categorical_imputer = (
            SimpleImputer(
                strategy="most_frequent"
            )
        )

        categorical_values = (
            categorical_imputer
            .fit_transform(
                df[categorical_cols]
            )
        )

        df[categorical_cols] = (
            pd.DataFrame(
                categorical_values,
                index=df.index,
                columns=categorical_cols
            )
        )

    # --------------------------------------------------------
    # NUMERICAL FEATURES
    # --------------------------------------------------------

    if numeric_cols:

        numeric_data = (
            df[numeric_cols]
            .copy()
        )

        # Initial deterministic fill

        initial = SimpleImputer(
            strategy="median"
        )

        initial_values = (
            initial.fit_transform(
                numeric_data
            )
        )

        filled = pd.DataFrame(
            initial_values,
            index=df.index,
            columns=numeric_cols
        )

        # Standardization

        scaler = StandardScaler()

        scaled = scaler.fit_transform(
            filled
        )

        # Missingness mask

        observed = (
            ~numeric_data.isna()
        )

        # Only train when enough numerical
        # features exist.

        if len(numeric_cols) >= 2:

            model = MLPRegressor(
                hidden_layer_sizes=(32,),
                activation="relu",
                solver="adam",
                learning_rate_init=0.001,
                max_iter=max_iter,
                random_state=random_state,
                early_stopping=True,
                validation_fraction=0.1
            )

            # Reconstruction target

            model.fit(
                scaled,
                scaled
            )

            reconstructed = (
                model.predict(
                    scaled
                )
            )

            reconstructed = scaler.inverse_transform(
                reconstructed
            )

            reconstructed_df = pd.DataFrame(
                reconstructed,
                index=df.index,
                columns=numeric_cols
            )

            # ------------------------------------------------
            # IMPORTANT:
            # Preserve all originally observed values.
            # ------------------------------------------------

            for column in numeric_cols:

                missing = (
                    numeric_data[column]
                    .isna()
                )

                df.loc[
                    missing,
                    column
                ] = reconstructed_df.loc[
                    missing,
                    column
                ]

        # Safety fallback

        for column in numeric_cols:

            if df[column].isna().any():

                df[column] = (
                    df[column]
                    .fillna(
                        df[column].median()
                    )
                )

    return df

In [14]:
# ============================================================
# 04.8 LLM-BASED BASELINE
# ============================================================

def impute_llm(
    masked_df,
    random_state=42,
    llm_callable=None
):
    """
    Standardized LLM-based imputation adapter.

    If llm_callable is supplied, it receives the masked
    DataFrame and must return an imputed DataFrame.

    If no LLM is supplied, a deterministic statistical
    fallback is used.

    This design keeps Notebook 04 reproducible and prevents
    external API calls from occurring inside every experiment.
    """

    df = masked_df.copy(
        deep=True
    )

    # --------------------------------------------------------
    # EXTERNAL LLM ADAPTER
    # --------------------------------------------------------

    if llm_callable is not None:

        result = llm_callable(
            df.copy(
                deep=True
            )
        )

        if not isinstance(
            result,
            pd.DataFrame
        ):

            raise TypeError(
                "LLM callable must return "
                "a pandas DataFrame."
            )

        return result

    # --------------------------------------------------------
    # DETERMINISTIC FALLBACK
    # --------------------------------------------------------

    numeric_cols = (
        df.select_dtypes(
            include=np.number
        )
        .columns
        .tolist()
    )

    categorical_cols = [
        column
        for column in df.columns
        if column not in numeric_cols
    ]

    if numeric_cols:

        numeric_imputer = (
            SimpleImputer(
                strategy="median"
            )
        )

        df[numeric_cols] = (
            numeric_imputer.fit_transform(
                df[numeric_cols]
            )
        )

    if categorical_cols:

        categorical_imputer = (
            SimpleImputer(
                strategy="most_frequent"
            )
        )

        df[categorical_cols] = (
            categorical_imputer.fit_transform(
                df[categorical_cols]
            )
        )

    return df

In [15]:
# ============================================================
# 04.9 AUTOMATED SELECTION BASELINE
# ============================================================

def select_automated_method(
    masked_df
):
    """
    Deterministic automated selection baseline.

    Selection uses only the observed structure of the
    incomplete dataset.

    No ground-truth values are used.
    No evaluation metrics are used.
    No missingness mask is used.

    Rules:
        - High-dimensional numerical data -> MICE
        - Mixed tabular data -> MICE
        - Small numerical datasets -> KNN
        - Otherwise -> Median/Mode
    """

    numeric_cols = (
        masked_df.select_dtypes(
            include=np.number
        )
        .columns
        .tolist()
    )

    categorical_cols = [
        column
        for column in masked_df.columns
        if column not in numeric_cols
    ]

    n_rows, n_columns = (
        masked_df.shape
    )

    missing_rate = (
        masked_df.isna()
        .mean()
        .mean()
    )

    # --------------------------------------------------------
    # DECISION RULES
    # --------------------------------------------------------

    if (
        len(numeric_cols) >= 5
        and missing_rate <= 0.30
    ):

        return "MICE"

    if (
        len(numeric_cols) >= 2
        and n_rows <= 10000
        and missing_rate <= 0.30
    ):

        return "KNN"

    if (
        len(categorical_cols) > 0
        and len(numeric_cols) > 0
    ):

        return "MICE"

    return "Median"

In [16]:
# ============================================================
# 04.9B AUTOMATED SELECTION IMPUTATION
# ============================================================

def impute_automated_selection(
    masked_df,
    random_state=42
):
    """
    Execute the automatically selected candidate.
    """

    selected_method = (
        select_automated_method(
            masked_df
        )
    )

    if selected_method == "MICE":

        result = impute_mice(
            masked_df,
            random_state=random_state
        )

    elif selected_method == "KNN":

        result = impute_knn(
            masked_df,
            random_state=random_state
        )

    else:

        result = impute_median(
            masked_df,
            random_state=random_state
        )

    return result

In [17]:
# ============================================================
# 04.10 CANDIDATE METHOD REGISTRY
# ============================================================

METHOD_REGISTRY = {

    "Mean": {

        "method_name":
            "Mean",

        "method_type":
            "statistical",

        "supported_feature_types":
            ["numerical", "categorical"],

        "hyperparameters": {},

        "fit":
            None,

        "transform":
            None,

        "predict_impute":
            impute_mean
    },

    "Median": {

        "method_name":
            "Median",

        "method_type":
            "statistical",

        "supported_feature_types":
            ["numerical", "categorical"],

        "hyperparameters": {},

        "fit":
            None,

        "transform":
            None,

        "predict_impute":
            impute_median
    },

    "Mode": {

        "method_name":
            "Mode",

        "method_type":
            "statistical",

        "supported_feature_types":
            ["numerical", "categorical"],

        "hyperparameters": {},

        "fit":
            None,

        "transform":
            None,

        "predict_impute":
            impute_mode
    },

    "KNN": {

        "method_name":
            "KNN",

        "method_type":
            "distance_based",

        "supported_feature_types":
            ["numerical", "categorical"],

        "hyperparameters": {
            "n_neighbors": 5,
            "weights": "distance"
        },

        "fit":
            None,

        "transform":
            None,

        "predict_impute":
            impute_knn
    },

    "MICE": {

        "method_name":
            "MICE",

        "method_type":
            "iterative_model_based",

        "supported_feature_types":
            ["numerical", "categorical"],

        "hyperparameters": {
            "max_iter": 10,
            "estimators": "ExtraTreesRegressor"
        },

        "fit":
            None,

        "transform":
            None,

        "predict_impute":
            impute_mice
    },

    "MissForest": {

        "method_name":
            "MissForest",

        "method_type":
            "ensemble_based",

        "supported_feature_types":
            ["numerical", "categorical"],

        "hyperparameters": {
            "n_estimators": 50,
            "max_iter": 5,
            "max_depth": 15,
            "n_jobs": -1
        },

        "fit":
            None,

        "transform":
            None,

        "predict_impute":
            impute_missforest
    },

    "GAIN": {

        "method_name":
            "GAIN",

        "method_type":
            "neural",

        "supported_feature_types":
            ["numerical", "categorical"],

        "hyperparameters": {
            "hidden_layer_sizes": [32],
            "max_iter": 20
        },

        "fit":
            None,

        "transform":
            None,

        "predict_impute":
            impute_gain
    },

    "LLM": {

        "method_name":
            "LLM",

        "method_type":
            "llm_based",

        "supported_feature_types":
            ["numerical", "categorical"],

        "hyperparameters": {
            "provider":
                "adapter",
            "temperature":
                0.0,
            "deterministic":
                True
        },

        "fit":
            None,

        "transform":
            None,

        "predict_impute":
            impute_llm
    },

    "AutomatedSelection": {

        "method_name":
            "AutomatedSelection",

        "method_type":
            "automated_selection",

        "supported_feature_types":
            ["numerical", "categorical"],

        "hyperparameters": {
            "selection":
                "structure_based",
            "uses_ground_truth":
                False
        },

        "fit":
            None,

        "transform":
            None,

        "predict_impute":
            impute_automated_selection
    }
}

METHOD_NAMES = list(
    METHOD_REGISTRY.keys()
)

print("=" * 90)
print("CANDIDATE METHOD REGISTRY")
print("=" * 90)

for method_name, config in (
    METHOD_REGISTRY.items()
):

    print(
        f"{method_name:<22}"
        f"{config['method_type']}"
    )

print()
print(
    f"Total candidate methods: "
    f"{len(METHOD_NAMES)}"
)

CANDIDATE METHOD REGISTRY
Mean                  statistical
Median                statistical
Mode                  statistical
KNN                   distance_based
MICE                  iterative_model_based
MissForest            ensemble_based
GAIN                  neural
LLM                   llm_based
AutomatedSelection    automated_selection

Total candidate methods: 9


In [18]:
# ============================================================
# 04.11 STANDARDIZED IMPUTATION INTERFACE
# ============================================================

def run_imputation(
    method_name,
    masked_df,
    missing_mask=None,
    random_state=42
):
    """
    Standardized AIR-LLM imputation interface.

    Parameters
    ----------
    method_name : str
        Candidate method name.

    masked_df : DataFrame
        Incomplete dataset.

    missing_mask : DataFrame, optional
        Experimental missingness mask.
        It is not used for model fitting.

    random_state : int
        Reproducibility seed.

    Returns
    -------
    dict
        Standardized experiment result.
    """

    if method_name not in METHOD_REGISTRY:

        raise KeyError(
            f"Unknown method: "
            f"{method_name}"
        )

    method = METHOD_REGISTRY[
        method_name
    ]

    imputer = method[
        "predict_impute"
    ]

    if imputer is None:

        raise ValueError(
            f"No imputation function registered "
            f"for {method_name}."
        )

    start_time = (
        time.perf_counter()
    )

    imputed_df = imputer(
        masked_df=masked_df.copy(
            deep=True
        ),
        random_state=random_state
    )

    runtime = (
        time.perf_counter()
        - start_time
    )

    # --------------------------------------------------------
    # STRUCTURAL VALIDATION
    # --------------------------------------------------------

    if not isinstance(
        imputed_df,
        pd.DataFrame
    ):

        raise TypeError(
            f"{method_name} did not return "
            f"a pandas DataFrame."
        )

    if imputed_df.shape != masked_df.shape:

        raise ValueError(
            f"{method_name} changed dataset shape."
        )

    if (
        imputed_df.columns.tolist()
        != masked_df.columns.tolist()
    ):

        raise ValueError(
            f"{method_name} changed column structure."
        )

    return {
        "method_name":
            method_name,

        "imputed_data":
            imputed_df,

        "runtime_seconds":
            float(runtime)
    }

In [19]:
# ============================================================
# 04.12 IMPUTATION OUTPUT VALIDATION
# ============================================================

def validate_imputation_output(
    masked_df,
    imputed_df,
    missing_mask=None
):
    """
    Validate structural integrity and completeness
    of an imputed dataset.
    """

    checks = {}

    checks[
        "same_shape"
    ] = (
        imputed_df.shape
        ==
        masked_df.shape
    )

    checks[
        "same_columns"
    ] = (
        imputed_df.columns.tolist()
        ==
        masked_df.columns.tolist()
    )

    checks[
        "no_missing_values"
    ] = (
        int(
            imputed_df.isna()
            .sum()
            .sum()
        ) == 0
    )

    checks[
        "no_infinite_values"
    ] = True

    numeric_cols = (
        imputed_df.select_dtypes(
            include=np.number
        ).columns
    )

    if len(numeric_cols) > 0:

        checks[
            "no_infinite_values"
        ] = (
            np.isfinite(
                imputed_df[
                    numeric_cols
                ]
                .to_numpy(
                    dtype=float
                )
            )
            .all()
        )

    # --------------------------------------------------------
    # OBSERVED VALUES MUST NOT CHANGE
    # --------------------------------------------------------

    checks[
        "observed_values_preserved"
    ] = True

    if missing_mask is not None:

        mask = (
            missing_mask
            .reindex(
                index=masked_df.index,
                columns=masked_df.columns
            )
            .fillna(False)
            .astype(bool)
        )

        observed = ~mask

        for column in masked_df.columns:

            original = (
                masked_df[column]
                .loc[observed[column]]
            )

            imputed = (
                imputed_df[column]
                .loc[observed[column]]
            )

            if not original.equals(
                imputed
            ):

                checks[
                    "observed_values_preserved"
                ] = False

                break

    checks[
        "valid"
    ] = all(
        checks.values()
    )

    checks[
        "remaining_missing_values"
    ] = int(
        imputed_df.isna()
        .sum()
        .sum()
    )

    return checks

In [20]:
# ============================================================
# 04.13 REGISTRY VALIDATION
# ============================================================

REQUIRED_REGISTRY_FIELDS = [
    "method_name",
    "method_type",
    "supported_feature_types",
    "hyperparameters",
    "fit",
    "transform",
    "predict_impute"
]

for method_name, config in (
    METHOD_REGISTRY.items()
):

    for field in REQUIRED_REGISTRY_FIELDS:

        assert field in config, (
            f"{method_name} is missing "
            f"registry field '{field}'."
        )

    assert callable(
        config["predict_impute"]
    ), (
        f"{method_name} does not have "
        f"a valid imputation function."
    )

print("=" * 90)
print("METHOD REGISTRY VALIDATED")
print("=" * 90)

print(
    f"Methods validated: "
    f"{len(METHOD_REGISTRY)}"
)

METHOD REGISTRY VALIDATED
Methods validated: 9


In [21]:
# ============================================================
# 04.14 CANDIDATE SMOKE TEST
# ============================================================

TEST_DATASET = (
    "adult_income"
)

TEST_SCENARIO = (
    "mcar_10pct_seed_42"
)

scenario_dir = (
    SCENARIO_ROOT
    / TEST_DATASET
)

scenario_path = (
    scenario_dir
    / f"{TEST_SCENARIO}.csv"
)

mask_path = (
    scenario_dir
    / f"{TEST_SCENARIO}_mask.csv"
)

if not scenario_path.exists():

    raise FileNotFoundError(
        f"Scenario not found:\n"
        f"{scenario_path}"
    )

if not mask_path.exists():

    raise FileNotFoundError(
        f"Mask not found:\n"
        f"{mask_path}"
    )

masked_df = pd.read_csv(
    scenario_path,
    low_memory=False
)

missing_mask = pd.read_csv(
    mask_path
).astype(bool)

# Align mask with data

missing_mask = (
    missing_mask
    .reindex(
        index=masked_df.index,
        columns=masked_df.columns
    )
    .fillna(False)
    .astype(bool)
)

print("=" * 90)
print("NOTEBOOK 04 — CANDIDATE SMOKE TEST")
print("=" * 90)

print(
    f"Dataset : {TEST_DATASET}"
)

print(
    f"Scenario: {TEST_SCENARIO}"
)

print(
    f"Shape   : {masked_df.shape}"
)

print(
    f"Missing cells: "
    f"{int(missing_mask.to_numpy().sum()):,}"
)

SMOKE_RESULTS = []

for method_name in METHOD_NAMES:

    print()
    print(
        f"Testing {method_name}..."
    )

    try:

        result = run_imputation(
            method_name,
            masked_df,
            missing_mask,
            random_state=RANDOM_STATE
        )

        validation = (
            validate_imputation_output(
                masked_df,
                result[
                    "imputed_data"
                ],
                missing_mask
            )
        )

        record = {

            "method":
                method_name,

            "runtime_seconds":
                result[
                    "runtime_seconds"
                ],

            "valid":
                validation[
                    "valid"
                ],

            "remaining_missing":
                validation[
                    "remaining_missing_values"
                ],

            "observed_values_preserved":
                validation[
                    "observed_values_preserved"
                ]
        }

        SMOKE_RESULTS.append(
            record
        )

        print(
            f"  Runtime : "
            f"{result['runtime_seconds']:.2f}s"
        )

        print(
            f"  Valid   : "
            f"{validation['valid']}"
        )

        print(
            f"  Missing : "
            f"{validation['remaining_missing_values']}"
        )

    except Exception as exc:

        SMOKE_RESULTS.append({

            "method":
                method_name,

            "runtime_seconds":
                np.nan,

            "valid":
                False,

            "remaining_missing":
                np.nan,

            "observed_values_preserved":
                False,

            "error":
                repr(exc)
        })

        print(
            f"  FAILED: "
            f"{repr(exc)}"
        )

SMOKE_DF = pd.DataFrame(
    SMOKE_RESULTS
)

print()
print("=" * 90)
print("SMOKE TEST SUMMARY")
print("=" * 90)

display(
    SMOKE_DF
)

NOTEBOOK 04 — CANDIDATE SMOKE TEST
Dataset : adult_income
Scenario: mcar_10pct_seed_42
Shape   : (19522, 14)
Missing cells: 27,331

Testing Mean...
  Runtime : 0.13s
  Valid   : True
  Missing : 0

Testing Median...
  Runtime : 0.11s
  Valid   : True
  Missing : 0

Testing Mode...
  Runtime : 0.10s
  Valid   : True
  Missing : 0

Testing KNN...
  Runtime : 15.86s
  Valid   : True
  Missing : 0

Testing MICE...
  Runtime : 17.52s
  Valid   : True
  Missing : 0

Testing MissForest...
  Runtime : 18.53s
  Valid   : True
  Missing : 0

Testing GAIN...
  Runtime : 0.89s
  Valid   : True
  Missing : 0

Testing LLM...
  Runtime : 0.06s
  Valid   : True
  Missing : 0

Testing AutomatedSelection...
  Runtime : 16.16s
  Valid   : True
  Missing : 0

SMOKE TEST SUMMARY


,method,runtime_seconds,valid,remaining_missing,observed_values_preserved
0,Mean,0.133449,True,0,True
1,Median,0.114209,True,0,True
2,Mode,0.098621,True,0,True
3,KNN,15.856671,True,0,True
4,MICE,17.517163,True,0,True
5,MissForest,18.530909,True,0,True
6,GAIN,0.890558,True,0,True
7,LLM,0.060501,True,0,True
8,AutomatedSelection,16.156041,True,0,True


In [22]:
# ============================================================
# 04.15 STRICT SMOKE-TEST GATE
# ============================================================

required_methods = set(
    METHOD_NAMES
)

tested_methods = set(
    SMOKE_DF["method"]
)

assert (
    tested_methods
    ==
    required_methods
), (
    "Not all candidate methods "
    "were tested."
)

failed_methods = (
    SMOKE_DF.loc[
        ~SMOKE_DF["valid"],
        "method"
    ]
    .tolist()
)

if failed_methods:

    raise RuntimeError(
        "Notebook 04 smoke test failed for:\n"
        + "\n".join(
            f"  - {method}"
            for method in failed_methods
        )
    )

assert (
    SMOKE_DF[
        "remaining_missing"
    ]
    .fillna(1)
    .eq(0)
    .all()
), (
    "At least one candidate left "
    "missing values."
)

assert (
    SMOKE_DF[
        "observed_values_preserved"
    ]
    .fillna(False)
    .all()
), (
    "At least one candidate modified "
    "observed values."
)

print("=" * 90)
print("NOTEBOOK 04 SMOKE TEST PASSED")
print("=" * 90)

print(
    f"Methods tested : {len(METHOD_NAMES)}"
)

print(
    "All candidates returned valid "
    "complete datasets."
)

print(
    "Observed values were preserved."
)

NOTEBOOK 04 SMOKE TEST PASSED
Methods tested : 9
All candidates returned valid complete datasets.
Observed values were preserved.


In [23]:
# ============================================================
# 04.16 SAVE CANDIDATE REGISTRY
# ============================================================

registry_records = []

for method_name, config in (
    METHOD_REGISTRY.items()
):

    registry_records.append({

        "method_name":
            config["method_name"],

        "method_type":
            config["method_type"],

        "supported_feature_types":
            json.dumps(
                config[
                    "supported_feature_types"
                ]
            ),

        "hyperparameters":
            json.dumps(
                config[
                    "hyperparameters"
                ],
                default=str
            ),

        "implementation":
            config[
                "predict_impute"
            ].__name__
    })

CANDIDATE_REGISTRY_DF = (
    pd.DataFrame(
        registry_records
    )
)

REGISTRY_PATH = (
    ARTIFACT_ROOT
    / "candidate_method_registry.csv"
)

CANDIDATE_REGISTRY_DF.to_csv(
    REGISTRY_PATH,
    index=False
)

SMOKE_PATH = (
    ARTIFACT_ROOT
    / "candidate_smoke_test.csv"
)

SMOKE_DF.to_csv(
    SMOKE_PATH,
    index=False
)

print("=" * 90)
print("NOTEBOOK 04 ARTIFACTS SAVED")
print("=" * 90)

print(
    f"Candidate registry:\n"
    f"{REGISTRY_PATH}"
)

print()

print(
    f"Smoke-test results:\n"
    f"{SMOKE_PATH}"
)

NOTEBOOK 04 ARTIFACTS SAVED
Candidate registry:
/content/drive/MyDrive/AIR_LLM_Research/artifacts/notebook_04/candidate_method_registry.csv

Smoke-test results:
/content/drive/MyDrive/AIR_LLM_Research/artifacts/notebook_04/candidate_smoke_test.csv


In [24]:
# ============================================================
# FINAL NOTEBOOK 04 — SAVE CANDIDATE STRATEGY REGISTRY
# ============================================================

from pathlib import Path
import json
import pandas as pd

print("=" * 90)
print("NOTEBOOK 04 — SAVING CANDIDATE STRATEGY REGISTRY")
print("=" * 90)

# ------------------------------------------------------------
# 1. CANONICAL OUTPUT DIRECTORY
# ------------------------------------------------------------

CANDIDATE_ROOT = (
    PROJECT_ROOT /
    "results" /
    "candidates"
)

CANDIDATE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

CANDIDATE_REGISTRY_PATH = (
    CANDIDATE_ROOT /
    "candidate_strategy_registry.csv"
)

CANDIDATE_REGISTRY_JSON_PATH = (
    CANDIDATE_ROOT /
    "candidate_strategy_registry.json"
)

# ------------------------------------------------------------
# 2. BUILD CANONICAL REGISTRY
# ------------------------------------------------------------

registry_records = []

for rank, method_name in enumerate(
    METHOD_NAMES,
    start=1
):

    registry_records.append({

        "method":
            str(method_name),

        "method_rank":
            int(rank),

        "registered":
            True,

        "smoke_tested":
            bool(
                method_name
                in tested_methods
            ),

        "smoke_test_valid":
            bool(
                SMOKE_DF.loc[
                    SMOKE_DF["method"]
                    == method_name,
                    "valid"
                ].all()
            )
    })

CANDIDATE_STRATEGY_REGISTRY_DF = pd.DataFrame(
    registry_records
)

# ------------------------------------------------------------
# 3. VALIDATE REGISTRY
# ------------------------------------------------------------

assert (
    set(
        CANDIDATE_STRATEGY_REGISTRY_DF[
            "method"
        ]
    )
    ==
    set(METHOD_NAMES)
), (
    "Candidate registry does not contain "
    "exactly the registered methods."
)

assert (
    CANDIDATE_STRATEGY_REGISTRY_DF[
        "registered"
    ].all()
), (
    "One or more candidate methods "
    "are not registered."
)

assert (
    CANDIDATE_STRATEGY_REGISTRY_DF[
        "smoke_tested"
    ].all()
), (
    "One or more candidate methods "
    "were not smoke tested."
)

assert (
    CANDIDATE_STRATEGY_REGISTRY_DF[
        "smoke_test_valid"
    ].all()
), (
    "One or more candidate methods "
    "failed the smoke test."
)

# ------------------------------------------------------------
# 4. SAVE CSV
# ------------------------------------------------------------

CANDIDATE_STRATEGY_REGISTRY_DF.to_csv(
    CANDIDATE_REGISTRY_PATH,
    index=False
)

# ------------------------------------------------------------
# 5. SAVE JSON
# ------------------------------------------------------------

candidate_registry_json = (
    CANDIDATE_STRATEGY_REGISTRY_DF
    .to_dict(orient="records")
)

with open(
    CANDIDATE_REGISTRY_JSON_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        candidate_registry_json,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 6. FINAL VERIFICATION
# ------------------------------------------------------------

assert CANDIDATE_REGISTRY_PATH.exists()
assert CANDIDATE_REGISTRY_JSON_PATH.exists()

print()
print("Registered candidate methods:")

for method in METHOD_NAMES:

    print(
        f"  {method:20s} "
        f"PASS"
    )

print()
print("=" * 90)
print("CANDIDATE STRATEGY REGISTRY SAVED")
print("=" * 90)

print(
    f"CSV : {CANDIDATE_REGISTRY_PATH}"
)

print(
    f"JSON: {CANDIDATE_REGISTRY_JSON_PATH}"
)

print()
print(
    f"Methods registered: "
    f"{len(CANDIDATE_STRATEGY_REGISTRY_DF)}"
)

print("=" * 90)

NOTEBOOK 04 — SAVING CANDIDATE STRATEGY REGISTRY

Registered candidate methods:
  Mean                 PASS
  Median               PASS
  Mode                 PASS
  KNN                  PASS
  MICE                 PASS
  MissForest           PASS
  GAIN                 PASS
  LLM                  PASS
  AutomatedSelection   PASS

CANDIDATE STRATEGY REGISTRY SAVED
CSV : /content/drive/MyDrive/AIR_LLM_Research/results/candidates/candidate_strategy_registry.csv
JSON: /content/drive/MyDrive/AIR_LLM_Research/results/candidates/candidate_strategy_registry.json

Methods registered: 9


In [25]:
# ============================================================
# 04.17 FINAL NOTEBOOK 04 VERIFICATION
# ============================================================

print("=" * 90)
print("FINAL NOTEBOOK 04 VERIFICATION")
print("=" * 90)

print()

print(
    f"Candidate methods : "
    f"{len(METHOD_REGISTRY)}"
)

print(
    f"Smoke-test methods: "
    f"{len(SMOKE_DF)}"
)

print()

for method_name in METHOD_NAMES:

    row = SMOKE_DF.loc[
        SMOKE_DF["method"]
        == method_name
    ]

    if len(row) != 1:

        raise AssertionError(
            f"Missing smoke-test result "
            f"for {method_name}"
        )

    status = bool(
        row.iloc[0]["valid"]
    )

    print(
        f"{method_name:<22} "
        f"{'PASS' if status else 'FAIL'}"
    )

print()

assert REGISTRY_PATH.exists()
assert SMOKE_PATH.exists()

print("=" * 90)
print("NOTEBOOK 04 COMPLETED SUCCESSFULLY")
print("=" * 90)

print(
    "Candidate methods are registered "
    "through the standardized interface."
)

print(
    "Notebook 04 does not execute the "
    "full 1,215-experiment matrix."
)

print(
    "The candidate layer is ready for "
    "the AIR-LLM experiment notebook."
)

FINAL NOTEBOOK 04 VERIFICATION

Candidate methods : 9
Smoke-test methods: 9

Mean                   PASS
Median                 PASS
Mode                   PASS
KNN                    PASS
MICE                   PASS
MissForest             PASS
GAIN                   PASS
LLM                    PASS
AutomatedSelection     PASS

NOTEBOOK 04 COMPLETED SUCCESSFULLY
Candidate methods are registered through the standardized interface.
Notebook 04 does not execute the full 1,215-experiment matrix.
The candidate layer is ready for the AIR-LLM experiment notebook.
